# Random Forest Regressor Model for Paris Airbnb Price Prediction
**Module**: IT3051 - Fundamentals of Data Mining  
**Group**: Noeva  
**Dataset**: Paris Detailed Listings (Inside Airbnb)  
**Task**: Supervised Machine Learning - Regression (Target: `log_price`)
---
This notebook demonstrates the complete implementation of the **Random Forest Regressor** model:
1. Loading preprocessed training and testing datasets
2. Training a Baseline Random Forest model
3. Model evaluation (R², RMSE, MAE in both log-scale and real Euro €)
4. Systematic Hyperparameter Tuning using `RandomizedSearchCV`
5. Evaluation of the Tuned Model & Performance Comparison
6. Feature Importance Analysis and Visualization
7. Exporting the trained model artifact (`.pkl`)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
import joblib

print("Libraries successfully imported!")

## 1. Load Preprocessed Data

In [ ]:
data_dir = "../data/processed"
X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv"))["log_price"]
y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv"))["log_price"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print(y_train.describe())

## 2. Evaluation Helper Function (Log and Euro € Scales)

In [ ]:
def evaluate_model(y_true_log, y_pred_log, name="Model"):
    r2 = r2_score(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    
    # Convert back to real Euros (€) using inverse log1p: expm1
    y_true_euros = np.expm1(y_true_log)
    y_pred_euros = np.expm1(y_pred_log)
    mae_euros = mean_absolute_error(y_true_euros, y_pred_euros)
    rmse_euros = np.sqrt(mean_squared_error(y_true_euros, y_pred_euros))
    
    print(f"=== {name} ===")
    print(f"R² Score:            {r2:.4f}")
    print(f"RMSE (Log scale):    {rmse_log:.4f}")
    print(f"MAE (Log scale):     {mae_log:.4f}")
    print(f"MAE (Real € / night): €{mae_euros:.2f}")
    print(f"RMSE (Real € / night): €{rmse_euros:.2f}\n")
    return {"Model": name, "R2": r2, "RMSE_log": rmse_log, "MAE_log": mae_log, "MAE_eur": mae_euros, "RMSE_eur": rmse_euros}

## 3. Train Baseline Random Forest Regressor

In [ ]:
rf_base = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)
res_base = evaluate_model(y_test, y_pred_base, name="Random Forest (Baseline)")

## 4. Hyperparameter Tuning using RandomizedSearchCV

In [ ]:
param_grid = {
    "n_estimators": [100, 150, 200],
    "max_depth": [15, 20, 25],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5, 0.8]
}

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=6,
    cv=3,
    scoring="neg_mean_squared_error",
    random_state=42,
    verbose=1,
    n_jobs=-1
)
search.fit(X_train, y_train)
print("Best Parameters Found:", search.best_params_)

## 5. Evaluate Tuned Model and Compare

In [ ]:
rf_tuned = search.best_estimator_
y_pred_tuned = rf_tuned.predict(X_test)
res_tuned = evaluate_model(y_test, y_pred_tuned, name="Random Forest (Tuned)")

comparison_df = pd.DataFrame([res_base, res_tuned])
display(comparison_df)

## 6. Feature Importance Analysis

In [ ]:
importances = rf_tuned.feature_importances_
top_indices = np.argsort(importances)[::-1][:15]
top_features = [X_train.columns[i] for i in top_indices]
top_scores = importances[top_indices]

plt.figure(figsize=(10, 6))
sns.barplot(x=top_scores, y=top_features, hue=top_features, palette="viridis", legend=False)
plt.title("Top 15 Most Important Listing Features Influencing Paris Airbnb Prices", fontsize=13, fontweight="bold")
plt.xlabel("Relative Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 7. Save Final Model Artifact

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(rf_tuned, "../models/random_forest_model.pkl")
print("Model saved to ../models/random_forest_model.pkl")